# Phase 1 - SQL Analytics Layer
## Healthcare AI system

**Objective:** Load raw CSVs into a relational SQLite database and run operational + financial analytics queries.
**Tables:**
- 'patients' - 5000 rows
- 'visits'   - 25000 rows
- 'billing'  - 25000 rows

In [1]:
# imports
import pandas as pd
import sqlite3
import os

In [2]:
# load raw csv files
patients = pd.read_csv("../data/patients.csv")
visits = pd.read_csv("../data/visits.csv")
billing = pd.read_csv("../data/billing.csv")

print("patients shape :",patients.shape)
print("visits shape :",visits.shape)
print("billing shape :",billing.shape)

patients shape : (5000, 7)
visits shape : (25000, 8)
billing shape : (25000, 7)


In [3]:
# create SQLite db
#creating db folder
os.makedirs("../db",exist_ok=True)

# connect to sqlite and create db
conn = sqlite3.connect("../db/hospital.db")
cursor = conn.cursor()
print("Database connected:",conn)

Database connected: <sqlite3.Connection object at 0x000002D87F7EC400>


In [4]:
# Loading the dataframes into sqlite tables
patients.to_sql("patients",conn,if_exists="replace",index=False)
visits.to_sql("visits",conn,if_exists="replace",index=False)
billing.to_sql("billing",conn,if_exists="replace",index=False)

print("All three tables loaded into hospital db")

All three tables loaded into hospital db


In [5]:
# quick preview of tables
print ("=== PATIENTS (first 3 rows) ===")
print(pd.read_sql("select * from patients limit 3",conn).to_string())

print ("=== VISITS (first 3 rows) ===")
print(pd.read_sql("select * from visits limit 3",conn).to_string())

print ("=== BILLING (first 3 rows) ===")
print(pd.read_sql("select * from billing limit 3",conn).to_string())

=== PATIENTS (first 3 rows) ===
   patient_id  age gender       city insurance_provider  chronic_flag registration_date
0           1   53      M  Hyderabad         SecureLife             0        2025-05-14
1           2   42      M       Pune         HealthPlus             0        2025-11-18
2           3   56      F  Hyderabad         HealthPlus             0        2025-05-11
=== VISITS (first 3 rows) ===
   visit_id  patient_id  visit_date   department visit_type  length_of_stay_hours risk_score  doctor_id
0         1         756  2025-10-18   Cardiology         ER                  3.48        Low        169
1         2        4102  2025-04-06  Orthopedics        OPD                 15.31       High        148
2         3        2964  2025-07-13          ICU         ER                 34.36        Low        153
=== BILLING (first 3 rows) ===
   bill_id  visit_id  billed_amount  approved_amount claim_status  payment_days billing_date
0        1         1       23577.37           

# Operational + Financial Analytics

In [6]:
dept_workload = pd.read_sql("""
SELECT 
    department,
    COUNT(visit_id)                     AS total_visits,
    ROUND(AVG(length_of_stay_hours), 2) AS avg_los_hours,
    ROUND(MAX(length_of_stay_hours), 2) AS max_los_hours,
    SUM(CASE WHEN risk_score = 'High'
        THEN 1 ELSE 0 END)              AS high_risk_visits
FROM visits
GROUP BY department
ORDER BY total_visits DESC
""",conn)
print(dept_workload.to_string(index=False))

 department  total_visits  avg_los_hours  max_los_hours  high_risk_visits
    General          4228          19.43          78.42               839
         ER          4220          19.53          74.81               872
  Neurology          4165          19.72          70.12               846
Orthopedics          4164          19.66          71.60               842
 Cardiology          4159          19.60          77.42               790
        ICU          4064          19.36          72.47               845


In [7]:
# Doctor Risk Cases
# which doctors handle the most high-risk patients
doctor_risk = pd.read_sql("""
SELECT
    doctor_id,
    COUNT(visit_id)                                 AS total_visits,
    SUM(CASE WHEN risk_score = 'High'
        THEN 1 ELSE 0 END)                          AS high_risk_cases,
    ROUND(
        100.0 * SUM(CASE WHEN risk_score = 'High'
        THEN 1 ELSE 0 END) / COUNT(visit_id), 1)    AS high_risk_pct
FROM visits
GROUP BY doctor_id
ORDER BY high_risk_cases DESC
LIMIT 10
""",conn)

print("Top 10 Doctors by high risk cases")
print(doctor_risk.to_string(index=False))

Top 10 Doctors by high risk cases
 doctor_id  total_visits  high_risk_cases  high_risk_pct
       174           264               71           26.9
       198           252               69           27.4
       169           279               68           24.4
       177           266               67           25.2
       135           261               65           24.9
       105           250               65           26.0
       188           285               64           22.5
       180           290               64           22.1
       131           266               62           23.3
       178           245               61           24.9


In [8]:
# Patients vitits patterns
# How many visits does each patient make on average
visit_patterns = pd.read_sql("""
SELECT
    visit_frequency_bucket,
    COUNT(patient_id) AS num_patients
FROM (
    SELECT
        patient_id,
        COUNT(visit_id) AS visit_count,
        CASE
            WHEN COUNT(visit_id) = 1 THEN '1 visit'
            WHEN COUNT(visit_id) BETWEEN 2 AND 3 THEN '2-3 visits' 
            WHEN COUNT(visit_id) BETWEEN 4 AND 5 THEN '4-5 visits' 
            ELSE '6+ visits'
        END AS visit_frequency_bucket
    FROM visits
    GROUP BY patient_id
)
GROUP BY visit_frequency_bucket
ORDER BY num_patients DESC
""",conn)

print("Patient Visit Frequency Distribution")
print(visit_patterns.to_string(index=False))

Patient Visit Frequency Distribution
visit_frequency_bucket  num_patients
             6+ visits          1937
            4-5 visits          1731
            2-3 visits          1150
               1 visit           149


In [9]:
# Risk Score Distribution by department
risk_by_dept = pd.read_sql("""
SELECT 
    department,
    SUM(CASE WHEN risk_score = 'Low' THEN 1 ELSE 0 END) AS low,
    SUM(CASE WHEN risk_score = 'Medium' THEN 1 ELSE 0 END) AS medium,
    SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) AS high,
    COUNT(*) AS total
FROM visits
GROUP BY department
ORDER BY high DESC

""",conn)
print("Risk Score Distribution by department")
print(risk_by_dept.to_string(index=False))

Risk Score Distribution by department
 department  low  medium  high  total
         ER 2092    1256   872   4220
  Neurology 2058    1261   846   4165
        ICU 2037    1182   845   4064
Orthopedics 2078    1244   842   4164
    General 2123    1266   839   4228
 Cardiology 2082    1287   790   4159


## Financial Analytics

In [10]:
# insurance Billing breakdown
# Revenue and claim outcomes by insurance provider
insurance_billing = pd.read_sql("""
SELECT 
    p.insurance_provider,
    COUNT(b.bill_id)                            AS total_claims,
    ROUND(SUM(b.billed_amount),0)               AS total_billed,
    ROUND(AVG(b.billed_amount),0)               AS avg_billed,
    ROUND(SUM(b.approved_amount),0)             AS total_approved,
    SUM(CASE WHEN b.claim_status = 'Paid' 
        THEN 1 ELSE 0 END)                      AS paid,
    SUM(CASE WHEN b.claim_status = 'Pending' 
        THEN 1 ELSE 0 END)                      AS pending,
    SUM(CASE WHEN b.claim_status = 'Rejected' 
        THEN 1 ELSE 0 END)                      AS rejected
FROM billing b
JOIN visits v ON b.visit_id = v.visit_id
JOIN patients p ON v.patient_id = p.patient_id
GROUP BY p.insurance_provider
ORDER BY total_billed DESC
""",conn)
print("Insurance Billing breakdown")
print(insurance_billing.to_string(index=False))

Insurance Billing breakdown
insurance_provider  total_claims  total_billed  avg_billed  total_approved  paid  pending  rejected
         MediCareX          6532   134591163.0     20605.0     100135469.0  3875     1661       996
           CareOne          6283   130707993.0     20803.0      96997758.0  3787     1562       934
        HealthPlus          6220   130180741.0     20929.0      96251775.0  3680     1609       931
        SecureLife          5965   126289040.0     21172.0      93770886.0  3598     1431       936


In [11]:
# Claim rejection analysis
# Which insurance providers reject the most claims?
rejection_analysis = pd.read_sql("""
SELECT
    p.insurance_provider,
    COUNT(b.bill_id)                            AS total_claims,
    SUM(CASE WHEN b.claim_status = 'Rejected' 
        THEN 1 ELSE 0 END)                      AS rejected_claims,
    ROUND(
        100/0 * SUM(CASE WHEN b.claim_status = 'Rejected' 
        THEN 1 ELSE 0 END) / COUNT(b.bill_id),1) AS rejection_rate_pct
FROM billing b
JOIN visits v on b.visit_id = v.visit_id
JOIN patients p on v.patient_id = v.patient_id
GROUP BY p.insurance_provider
ORDER BY rejection_rate_pct DESC
""",conn)

print("Claim rejection rate by insurance provider")
print(rejection_analysis.to_string(index=False))

Claim rejection rate by insurance provider
insurance_provider  total_claims  rejected_claims rejection_rate_pct
        SecureLife      30000000          4556400               None
         MediCareX      32200000          4890536               None
        HealthPlus      31250000          4746250               None
           CareOne      31550000          4791814               None


In [13]:
# Rvenue Realization
# how much of what we bill actually gets approved
revenue_realization = pd.read_sql("""
SELECT
    p.insurance_provider,
    ROUND(SUM(b.billed_amount),0)           AS total_billed,
    ROUND(SUM(b.approved_amount),0)         AS total_approved,
    ROUND(
        100.0 * SUM(b.approved_amount) / 
        SUM(b.billed_amount),1)             AS realization_rate_pct
FROM billing b
JOIN visits v on b.visit_id = v.visit_id
JOIN patients p on v.patient_id = p.patient_id
WHERE b.approved_amount IS NOT NULL
GROUP BY p.insurance_provider
ORDER BY realization_rate_pct DESC
""",conn)
print("Revenue realization rate by insurance provider")
print(revenue_realization.to_string(index=False))

Revenue realization rate by insurance provider
insurance_provider  total_billed  total_approved  realization_rate_pct
           CareOne   123701662.0      96997758.0                  78.4
        HealthPlus   122873457.0      96251775.0                  78.3
         MediCareX   128114983.0     100135469.0                  78.2
        SecureLife   120079039.0      93770886.0                  78.1
